# **Transcriptional Factor (TF)-Target Analysis for Microglia**

This notebook provides an analytical framework for processing the pan-dataset TF-target interaction matrix derived from hdWGCNA. It includes tools to validate specific TF-target pairs, identify top upstream regulators for specific genes, and characterize downstream targets for selected transcription factors across microglial datasets.

In [ ]:
# Import libraries

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Read the DataFrame

merged_df = pd.read_csv('TF_Target_Model_for_Microglia.csv')

In [ ]:
merged_df

The final integrated model contains 2,515,157 unique TF-target pairs. The primary variables of interest are the `Cor_dsX` columns, which represent the XGBoost regulation coefficients for each specific dataset. These coefficients range from -1 to +1, where negative values indicate transcriptional repression and positive values indicate transcriptional activation. A value of `NaN` signifies the absence of a predicted interaction for that pair within the respective dataset's model.

# **Exploring one TF-Target Pair**

In [ ]:
def plot_tf_target_regulation(df, tf_name, target_gene):
    """
    Plots a 3D-style horizontal bar plot for a given TF-target gene regulation pair.
    Highlights the mean correlation across all 9 cross-species datasets and individual data points.
    """
    # Local constants to prevent NameError
    cor_cols = [f'Cor_ds{i}' for i in range(1, 10)]
    
    dataset_labels = {
        'DS1': 'DS1 (Human)', 'DS2': 'DS2 (Human)', 'DS3': 'DS3 (Human)',
        'DS4': 'DS4 (Human)', 'DS5': 'DS5 (Human)',
        'DS6': 'DS6 (M. fascicularis)', 'DS7': 'DS7 (M. mulatta)',
        'DS8': 'DS8 (Human)', 'DS9': 'DS9 (Human)'
    }
    
    dataset_palette = {
        'DS1': '#8A6CF4', 'DS2': '#88510B', 'DS3': '#C96BF5',
        'DS4': '#6CC3F4', 'DS5': '#EEF46C', 'DS6': '#b5e716',
        'DS7': '#F4736C', 'DS8': '#4C9BE8', 'DS9': '#F46CB8'
    }

    # 1. Data retrieval and preparation
    pair_data = df[(df['tf'] == tf_name) & (df['gene'] == target_gene)]

    if pair_data.empty:
        print(f"Error: Pair {tf_name} -> {target_gene} not found in the table.")
        return

    # Extract correlation values across the 9 datasets
    values_series = pair_data[cor_cols].iloc[0]

    # Separate present and missing data points
    present_ds = values_series.dropna()
    missing_labels = [col.replace('Cor_ds', 'DS') for col in values_series[values_series.isna()].index]

    plot_df = pd.DataFrame({
        'Dataset': [col.replace('Cor_ds', 'DS') for col in present_ds.index],
        'Correlation': present_ds.values,
        'jitter': np.random.uniform(-0.08, 0.08, size=len(present_ds))
    })
    plot_df['Label'] = plot_df['Dataset'].map(dataset_labels)

    mean_val = present_ds.mean() if not present_ds.empty else 0

    # 2. Visualization style setup
    sns.set_style("white")
    fig, ax = plt.subplots(figsize=(12, 5))

    # Colorblind-friendly palette selection: mint for activation, fuchsia for repression
    bar_color = '#8ef689' if mean_val > 0 else '#f688de'

    # Draw the 3D-style bar (layering transparent bars)
    if not present_ds.empty:
        for i in range(1, 15):
            ax.barh(0, mean_val, height=0.3, color=bar_color, alpha=0.06 * i, zorder=2)
        # Shadow effect
        ax.barh(-0.01, mean_val, height=0.3, color='black', alpha=0.1, zorder=1)

    # 3. Scatter points for individual datasets
    for _, row in plot_df.iterrows():
        ax.scatter(row['Correlation'], row['jitter'],
                   color=dataset_palette.get(row['Dataset'], 'grey'), s=280,
                   edgecolor='black', linewidth=1.5, label=row['Label'],
                   zorder=5, alpha=0.9)

    # 4. Axis configuration and styling
    ax.axvline(0, color='#333333', linewidth=2, zorder=3)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-0.4, 0.4)

    # Set y-axis ticks and labels
    ax.set_yticks([0])
    label_text = f"{tf_name} → {target_gene}"
    ax.set_yticklabels([label_text], fontsize=14, fontweight='bold', fontstyle='italic')

    ax.set_xticks(np.arange(-1, 1.1, 0.2))
    ax.grid(axis='x', linestyle='--', alpha=0.4)

    # 5. Annotation for missing data
    info_text = "No data from: " + (", ".join(missing_labels) if missing_labels else "None")
    plt.figtext(0.5, -0.02, info_text, ha="center", fontsize=11,
                bbox={"facecolor":"#FAD7A0", "alpha":0.2, "pad":5}, color="#5D6D7E")

    # Set the main title and legend
    title_text = f'Regulation Analysis (Mean Cor: {mean_val:.3f})'
    ax.set_title(title_text, fontsize=16, pad=20, fontstyle='italic')
    ax.legend(title='Datasets', bbox_to_anchor=(1.02, 1), loc='upper left')

    sns.despine(left=True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_tf_target_regulation(merged_df, 'MEF2A', 'P2RY12')

MEF2A activates P2RY12.

In [ ]:
plot_tf_target_regulation(merged_df, 'PPARG', 'SPP1')

PPARG activates SPP1.

In [ ]:
plot_tf_target_regulation(merged_df, 'MEF2C', 'GPNMB')

MEF2C represses GPNMB

In [ ]:
plot_tf_target_regulation(merged_df, 'MITF', 'P2RY12')

There is no direct regulation of P2RY12 by MITF (likely because the MITF sequence is absent in the promoter of P2RY12).

# **Find top regulators for a gene**

In [ ]:
def plot_target_top_tfs(df, target_gene, n_activate=5, m_repress=5, k=3):
    """
    Identifies and plots the top activating and repressing transcription factors (TFs) 
    regulating a specific target gene based on cross-dataset mean correlation.
    """
    # 1. Data filtering and thresholding
    gene_df = df[df['gene'] == target_gene].copy()
    if gene_df.empty:
        print(f"Error: Target gene '{target_gene}' not found.")
        return

    # Identify dynamic correlation columns across datasets
    cor_cols = [c for c in df.columns if c.startswith('Cor_ds')]
    
    # Filter TFs based on presence in at least 'k' datasets
    gene_df['n_datasets'] = gene_df[cor_cols].notna().sum(axis=1)
    gene_df = gene_df[gene_df['n_datasets'] >= k].copy()
    
    if gene_df.empty:
        print(f"No TFs pass the minimum dataset threshold (k >= {k}) for {target_gene}.")
        return

    # Calculate global mean correlation for ranking
    gene_df['Mean_Cor'] = gene_df[cor_cols].mean(axis=1)

    # 2. Select top upstream regulators
    top_activators = gene_df.sort_values(by='Mean_Cor', ascending=False).head(n_activate)
    top_repressors = gene_df.sort_values(by='Mean_Cor', ascending=True).head(m_repress)
    plot_data = pd.concat([top_repressors, top_activators]).sort_values(by='Mean_Cor', ascending=True).drop_duplicates().reset_index(drop=True)

    # 3. Canvas geometry layout
    height_per_row = 0.5
    total_height = len(plot_data) * height_per_row + 2.5 

    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=(12, total_height), dpi=600)

    # Palette definition (mint for activation, fuchsia for repression)
    bar_color_pos, bar_color_neg = '#8ef689', '#f688de'
    
    dataset_palette = {
        'DS1': '#8A6CF4', 'DS2': '#88510B', 'DS3': '#C96BF5', 'DS4': '#6CC3F4', 'DS5': '#EEF46C',
        'DS6': '#b5e716', 'DS7': '#F4736C', 'DS8': '#4C9BE8', 'DS9': '#F46CB8' 
    }

    y_positions = np.arange(len(plot_data))

    # 4. Rendering logic (bars and scatter points)
    for idx, (_, row) in enumerate(plot_data.iterrows()):
        mean_val = row['Mean_Cor']
        
        # Plot heavy horizontal bars
        ax.barh(y_positions[idx], mean_val, 
                color=(bar_color_pos if mean_val > 0 else bar_color_neg),
                height=0.7, alpha=0.9, edgecolor='black', linewidth=1.2, zorder=2)

        # Plot individual dataset data points with jitter
        present_values = row[cor_cols].dropna()
        for col, val in present_values.items():
            ds_key = f'DS{col.replace("Cor_ds", "")}'
            ax.scatter(val, y_positions[idx] + np.random.uniform(-0.15, 0.15),
                       color=dataset_palette.get(ds_key, 'grey'), s=180, 
                       edgecolor='black', linewidth=0.8, zorder=5)

    # 5. Typography and axis formatting
    ax.axvline(0, color='#333333', linewidth=2, zorder=3)
    ax.set_xlim(-1.05, 1.05)

    # Configure axes labels
    ax.set_xlabel('Mean Correlation Coefficient', fontsize=21, fontweight='bold')
    ax.set_ylabel('Transcription Factor (TF)', fontsize=21, fontweight='bold')
    
    # Configure y-ticks with bold-italic gene identifiers
    ax.set_yticks(y_positions)
    ax.set_yticklabels(plot_data['tf'].tolist(), fontsize=23, fontstyle='italic', fontweight='bold')
    
    # Configure x-axis ticks font size
    ax.tick_params(axis='x', labelsize=19)

    # Main manuscript title configuration
    ax.set_title(f'Top TFs Regulating Target Gene: {target_gene}',
                 fontsize=25, pad=30, fontstyle='italic', fontweight='bold')

    # 6. Manuscript-ready legend layout
    all_ds_keys = [f'DS{i}' for i in range(1, 10)]
    handles = [plt.Line2D([], [], marker='o', color='w', label=ds,
                          markerfacecolor=dataset_palette.get(ds, 'grey'), 
                          markersize=15, markeredgecolor='black')
               for ds in all_ds_keys]

    ax.legend(handles=handles, title='Datasets', title_fontsize=23,
              fontsize=20, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_target_top_tfs(merged_df, 'SORL1', n_activate=5, m_repress=5)

In [ ]:
plot_target_top_tfs(merged_df, 'FRMD4A')

Both SORL1 and FRMD4A are homeostatic genes in our analysis. Both are activated by MEF2A and MEF2C, and repressed by RBPJ, which are all among their top regulators.

In [ ]:
plot_target_top_tfs(merged_df, 'GPNMB')

In [ ]:
plot_target_top_tfs(merged_df, 'DPYD')

Both GPNMB and DPYD are critical players of microglial activation. They are both activated by broad (HIF1A, RBPJ) and more specific (such as MITF, PPARG, FOXP1) TFs, and simultaneously repressed by pro-homeostatic TFs (including MEF2A, MEF2C, MEIS1). 

# **Find top targets for a given TF**

In [ ]:
def plot_tf_top_targets(df, tf_name, n_activate=5, m_repress=5, k=3):
    """
    Identifies and plots the top downstream target genes (activated and repressed) 
    regulated by a specific transcription factor (TF) based on cross-dataset mean correlation.
    """
    # 1. Data filtering and thresholding
    tf_df = df[df['tf'] == tf_name].copy()
    if tf_df.empty:
        print(f"Error: Transcription Factor '{tf_name}' not found.")
        return

    # Identify dynamic correlation columns across datasets
    cor_cols = [c for c in df.columns if c.startswith('Cor_ds')]
    
    # Filter targets based on presence in at least 'k' datasets
    tf_df['n_datasets'] = tf_df[cor_cols].notna().sum(axis=1)
    tf_df = tf_df[tf_df['n_datasets'] >= k].copy()
    
    if tf_df.empty:
        print(f"Error: No targets found for TF '{tf_name}' with at least {k} datasets.")
        return

    # Calculate global mean correlation for ranking
    tf_df['Mean_Cor'] = tf_df[cor_cols].mean(axis=1)

    # 2. Select top downstream targets
    top_activators = tf_df.sort_values(by='Mean_Cor', ascending=False).head(n_activate)
    top_repressors = tf_df.sort_values(by='Mean_Cor', ascending=True).head(m_repress)
    plot_data = pd.concat([top_repressors, top_activators]).sort_values(by='Mean_Cor', ascending=True).drop_duplicates().reset_index(drop=True)

    # 3. Canvas geometry layout
    height_per_row = 0.5
    total_height = len(plot_data) * height_per_row + 2.5 

    sns.set_style("whitegrid")
    fig, ax = plt.subplots(figsize=(12, total_height), dpi=600)

    # Palette definition (mint for activation, fuchsia for repression)
    bar_color_pos, bar_color_neg = '#8ef689', '#f688de'
    
    dataset_palette = {
        'DS1': '#8A6CF4', 'DS2': '#88510B', 'DS3': '#C96BF5', 'DS4': '#6CC3F4', 'DS5': '#EEF46C',
        'DS6': '#b5e716', 'DS7': '#F4736C', 'DS8': '#4C9BE8', 'DS9': '#F4B86C' 
    }

    y_positions = np.arange(len(plot_data))

    # 4. Rendering logic (bars and scatter points)
    for idx, (_, row) in enumerate(plot_data.iterrows()):
        mean_val = row['Mean_Cor']
        
        # Plot heavy horizontal bars
        ax.barh(y_positions[idx], mean_val, 
                color=(bar_color_pos if mean_val > 0 else bar_color_neg),
                height=0.7, alpha=0.9, edgecolor='black', linewidth=1.2, zorder=2)

        # Plot individual dataset data points with jitter
        present_values = row[cor_cols].dropna()
        for col, val in present_values.items():
            ds_key = f'DS{col.replace("Cor_ds", "")}'
            ax.scatter(val, y_positions[idx] + np.random.uniform(-0.15, 0.15),
                       color=dataset_palette.get(ds_key, 'grey'), s=180, 
                       edgecolor='black', linewidth=0.8, zorder=5)

    # 5. Typography and axis formatting
    ax.axvline(0, color='#333333', linewidth=2, zorder=3)
    ax.set_xlim(-1.05, 1.05)

    # Configure axes labels
    ax.set_xlabel('Mean Correlation Coefficient', fontsize=21, fontweight='bold')
    ax.set_ylabel('Target Gene', fontsize=21, fontweight='bold')
    
    # Configure y-ticks with bold-italic gene identifiers
    ax.set_yticks(y_positions)
    ax.set_yticklabels(plot_data['gene'].tolist(), fontsize=23, fontstyle='italic', fontweight='bold')
    
    # Configure x-axis ticks font size
    ax.tick_params(axis='x', labelsize=19)

    # Main manuscript title configuration
    ax.set_title(f'Top Targets Regulated By: {tf_name}',
                 fontsize=25, pad=30, fontstyle='italic', fontweight='bold')

    # 6. Manuscript-ready legend layout
    all_ds_keys = [f'DS{i}' for i in range(1, 10)]
    handles = [plt.Line2D([], [], marker='o', color='w', label=ds,
                          markerfacecolor=dataset_palette.get(ds, 'grey'), 
                          markersize=15, markeredgecolor='black')
               for ds in all_ds_keys]

    ax.legend(handles=handles, title='Datasets', title_fontsize=23,
              fontsize=20, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_tf_top_targets(merged_df, 'MITF')

In [ ]:
plot_tf_top_targets(merged_df, 'PPARG', n_activate=7)

In [ ]:
plot_tf_top_targets(merged_df, 'FOXP1')

All three TFs are highly important for microglial activation. MITF activates GPNMB, MYO1E, and STARD13. PPARG co-activates GPNMB and SPP1. FOXP1 activates LRRK2, DPYD, and TANC2, and represses MEIS1.

In [ ]:
plot_tf_top_targets(merged_df, 'MEF2C')

In [ ]:
plot_tf_top_targets(merged_df, 'FOXP2')

In [ ]:
plot_tf_top_targets(merged_df, 'MEIS1')

These are critically important pro-homeostatic TFs. MEF2C, FOXP2, and several others homogeneously activate the expression of several critical homeostatic genes. MEIS1 is a master repressor, particularly for APOE.